# Dataset Size Profiling Plot

Reads benchmark files from `dataset_size/data` and plots CPU vs GPU runtime by repeat factor.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

base_dir = Path.cwd()
if not (base_dir / 'data').exists():
    candidate = (Path.cwd() / 'notebooks' / 'profiling' / 'dataset_size').resolve()
    if (candidate / 'data').exists():
        base_dir = candidate

data_dir = base_dir / 'data'
csv_path = data_dir / 'dataset_size_runs_latest.csv'
if not csv_path.exists():
    raise FileNotFoundError(f'Missing {csv_path}. Run run_dataset_size.py first.')

df = pd.read_csv(csv_path)
df['repeat_factor'] = pd.to_numeric(df['repeat_factor'])
df['elapsed_seconds'] = pd.to_numeric(df['elapsed_seconds'])
df['mode_label'] = np.where(df['use_gpu'], 'GPU bitset', 'CPU bitset')
df.head()


In [ ]:
summary = (
    df.groupby(['mode_label', 'repeat_factor'], as_index=False)
      .agg(
          runs=('elapsed_seconds', 'count'),
          mean_s=('elapsed_seconds', 'mean'),
          median_s=('elapsed_seconds', 'median'),
          std_s=('elapsed_seconds', lambda x: float(x.std(ddof=1)) if len(x) > 1 else 0.0),
          rules_min=('rule_count', 'min'),
          rules_max=('rule_count', 'max'),
      )
      .sort_values(['mode_label', 'repeat_factor'])
)
summary


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for mode, g in summary.groupby('mode_label'):
    ax.errorbar(
        g['repeat_factor'],
        g['mean_s'],
        yerr=g['std_s'],
        marker='o',
        linewidth=2,
        capsize=4,
        label=mode,
    )
    for _, row in g.iterrows():
        ax.annotate(
            f"{row['mean_s']:.3f} +/- {row['std_s']:.3f}",
            xy=(row['repeat_factor'], row['mean_s'] + row['std_s']),
            xytext=(0, 6),
            textcoords='offset points',
            ha='center',
            va='bottom',
            fontsize=8,
        )

ax.set_xlabel('Repeat factor')
ax.set_ylabel('Runtime mean (s)')
ax.set_title('CPU vs GPU bitset runtime by dataset size')
ax.grid(alpha=0.25)
ax.legend()

plots_dir = base_dir / 'plots'
plots_dir.mkdir(parents=True, exist_ok=True)
out_png = plots_dir / 'runtime_vs_repeat_with_std.png'
plt.tight_layout()
plt.savefig(out_png, dpi=300, format='png')
print('Saved:', out_png)
plt.show()
